# 监控指标清单的核心逻辑是：
- 以训练前100~500步的分位数作为健康基线，<font color='red'>重点监控相对偏移而非绝对数值</font>；
  - 建议直接在训练代码中每N步记录并绘制p5/p50/p95三条曲线，三条线紧密贴合且平稳即为健康，发散或整体下移趋近零则触发告警。

## 核心实现逻辑
1. Warmup 阶段（前 N 步）： 只收集数据，不报警。结束后计算 P5/P50/P95 作为初始健康基线。
2. 监控阶段（N 步后）：
  - 计算当前步的梯度范数。
  - 相对偏移检查： 如果当前值偏离基线中位数（P50）超过一定倍数（如 3 倍），触发警报。
  - 实时统计更新： 持续记录数据，计算实时的 P5/P50/P95，用于绘制曲线观察发散情况。

np.percentile 的参数 q 范围是 [0, 100]，而 PyTorch 的 torch.quantile 参数 q 范围是 [0, 1]，两者本质相同，只是单位不同。

| NumPy | PyTorch | 说明 |
|-------|---------|------|
| `np.percentile(x, 50)` | `torch.quantile(x, 0.5)` | 中位数 |
| `np.percentile(x, 5)` | `torch.quantile(x, 0.05)` | P5 |
| `np.percentile(x, 95)` | `torch.quantile(x, 0.95)` | P95 |
| `np.nanpercentile(x, 50)` | `torch.nanquantile(x, 0.5)` | 忽略 NaN 的中位数 |



In [ ]:
import torch
import torch.nn as nn
import numpy as np
from collections import deque

class GradientHealthMonitor:
    def __init__(self, warmup_steps=100, spike_threshold=3.0):
        """
        Args:
            warmup_steps: 用于建立基线的步数 (建议 100-500)
            spike_threshold: 触发尖峰警报的倍数 (相对于基线 P50)
        """
        self.warmup_steps = warmup_steps
        self.spike_threshold = spike_threshold
        
        self.step = 0
        self.history = []  # 存储所有历史梯度范数
        
        # 基线指标 (在 warmup 结束后填充)
        self.baseline_p50 = None 
        self.baseline_p5 = None
        self.baseline_p95 = None
        
        self.is_warmup_finished = False

    def _compute_norm(self, model):
        """计算全局 L2 梯度范数"""
        total_norm = 0.0
        count = 0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
                count += 1
        
        # 防止除零或无梯度情况
        if count == 0: return 0.0
        total_norm = total_norm ** 0.5
        return total_norm

    def update(self, model):
        """
        在 optimizer.step() 之前调用
        Returns: dict 包含状态信息
        """
        grad_norm = self._compute_norm(model)
        self.history.append(grad_norm)
        self.step += 1
        
        status = {
            "step": self.step,
            "grad_norm": grad_norm,
            "state": "WARMUP",
            "alert": False,
            "msg": ""
        }

        # 1. Warmup 阶段结束，确立基线
        if self.step == self.warmup_steps:
            arr = np.array(self.history)
            
            self.baseline_p50 = np.percentile(arr, 50)
            self.baseline_p5 = np.percentile(arr, 5)
            self.baseline_p95 = np.percentile(arr, 95)
            
            self.is_warmup_finished = True
            
            status["state"] = "BASELINE_ESTABLISHED"
            status["msg"] = f"基线已建立 | P50={self.baseline_p50:.4f}, P5-P95=[{self.baseline_p5:.4f}-{self.baseline_p95:.4f}]"
            print(f"\n[Monitor] {status['msg']}")

        # 2. 监控阶段
        elif self.is_warmup_finished:
            status["state"] = "MONITORING"
            
            # --- 核心逻辑 A: 检查相对偏移 (尖峰检测) ---
            # 如果当前值超过基线中位数的 threshold 倍
            if self.baseline_p50 > 1e-8: # 防止基线为0
                ratio = grad_norm / self.baseline_p50
                if ratio > self.spike_threshold:
                    status["alert"] = True
                    status["msg"] = f"⚠️ 梯度尖峰! 当前:{grad_norm:.4f} (是基线P50的 {ratio:.1f} 倍)"
            
            # --- 核心逻辑 B: 检查发散 (可选，用于绘图分析) ---
            # 如果需要在训练中实时看 P5/P50/P95 曲线是否发散
            # 注意：全量计算百分位比较耗时，生产环境建议每 N 步算一次或用近似算法
            if self.step % 50 == 0:
                recent_arr = np.array(self.history[-100:]) # 取最近100步看局部趋势
                cur_p50 = np.percentile(recent_arr, 50)
                
                '''
                
                / (self.baseline_p50 + 1e-8)
                - 归一化（关键步骤）：
                - 将绝对差值除以基线值，得到一个相对比率。
                - 为什么要这样做？ 
                   - 因为不同模型的梯度范数量级差异巨大。有的模型梯度是 0.01，有的是 100。
                   - 如果不做归一化，你就无法设定一个通用的报警阈值（比如 0.5）。
                '''
                # 简单的发散检查：当前的局部中位数是否已经严重偏离了初始基线
                drift_ratio = abs(cur_p50 - self.baseline_p50) / (self.baseline_p50 + 1e-8)
                
                # 2. 触发阈值判断
                if drift_ratio > 0.5: # 比如漂移超过 50%
                     status["msg"] += f" | ⚠️ 趋势漂移警告: 局部P50已偏离基线 {drift_ratio:.1%}"

        return status

# ==========================================
# 模拟训练演示
# ==========================================
def dummy_train():
    # 初始化一个简单的模型
    model = nn.Linear(10, 10)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    
    # 初始化监控器：前 100 步学习基线，超过 3 倍报警
    monitor = GradientHealthMonitor(warmup_steps=100, spike_threshold=3.0)
    
    print("开始模拟训练...")
    
    for i in range(300):
        optimizer.zero_grad()
        
        # 模拟数据
        x = torch.randn(32, 10)
        y = torch.randn(32, 10)
        loss = ((model(x) - y)**2).mean()
        
        # 模拟异常：在第 200 步人为制造一个巨大的 Loss (模拟脏数据或 Reward 爆炸)
        if i == 200:
            loss = loss * 500 
        
        loss.backward()
        
        # 【关键】在 step 之前检查梯度
        info = monitor.update(model)
        
        # 打印日志
        if info["state"] != "MONITORING" or info["alert"] or i % 50 == 0:
            print(f"Step {info['step']:4d} | Norm: {info['grad_norm']:.4f} | {info['state']} | {info['msg']}")
            
        # 只有没报警才更新参数 (或者你可以选择 Clip 后更新)
        if not info["alert"]:
            optimizer.step()
        else:
            print(f"       -> 跳过本次参数更新以保护模型")

if __name__ == "__main__":
    dummy_train()

具体清单与阈值如下：

## 奖励分布监控

| 统计量 | 健康范围 | 警告信号 | 诊断含义 |
|--------|----------|----------|----------|
| 奖励均值 μ | 随训练缓慢上升 | 持续下降或剧烈震荡 | 策略退化或奖励信号不稳定 |
| 奖励标准差 σ | > 0.1 | σ → 0 | 模型输出同质化，可能熵坍塌 |
| p5 ~ p95 区间宽度 | 占全量程 30%~70% | 区间极窄（< 10%） | 零方差组占比过高 |
| p99 / p50 比值 | < 3 | > 5 | 长尾异常值驱动训练 |
| 零方差组占比 | < 10% | > 30% | 大量样本无法提供学习信号 |



## 概率比（Ratio）监控

| 统计量 | 健康范围 | 警告信号 | 诊断含义 |
|--------|----------|----------|----------|
| ratio 中位数 p50 | 0.9 ~ 1.1 | 持续偏离 1.0 | 策略更新幅度过大或过小 |
| ratio p5 ~ p95 | [0.8, 1.2] | 超出 [0.5, 2.0] | 新旧策略分歧过大 |
| ratio p99 | < 3.0 | > 10 | 数值溢出风险，低精度下尤其危险 |
| ratio p1 | > 0.3 | < 0.1 | 新策略完全抛弃旧策略的某些行为 |



## 优势值（Advantage）监控

| 统计量 | 健康范围 | 警告信号 | 诊断含义 |
|--------|----------|----------|----------|
| 优势值均值 | ≈ 0（±0.1） | 持续偏离 0 | 优势估计实现有误或奖励基线漂移 |
| 优势值标准差 | > 0.05 | σ → 0 | 组内无差异，梯度消失 |
| p90 / \|p10\| 比值 | 0.5 ~ 2.0 | > 5 或 < 0.2 | 正负样本严重不对称 |
| 优势值 p99 | < 5.0 | > 10 | 极端样本绑架梯度方向 |



## KL 散度监控

| 统计量 | 健康范围 | 警告信号 | 诊断含义 |
|--------|----------|----------|----------|
| KL 均值 | 0.001 ~ 0.03 | > 0.05 | 策略偏离参考模型过远 |
| KL p95 | < 0.05 | 持续攀升 | 过优化风险，需增大 KL 系数 |
| KL p50 趋势 | 平稳或缓慢上升 | 指数级上升 | 策略失控，训练即将崩溃 |



## 梯度范数监控

| 统计量 | 健康范围 | 警告信号 | 诊断含义 |
|--------|----------|----------|----------|
| 梯度范数 p50 | 0.1 ~ 1.0 | < 0.01 或 > 5.0 | 梯度消失或爆炸 |
| 梯度范数 p99 | < 5.0 | > 10 | 存在梯度爆炸风险 |
| 梯度范数变异系数 | < 0.5 | > 1.0 | 训练极不稳定，需降低学习率 |
| 连续 NaN step 数 | 0 | > 0 | 数值溢出，立即排查 |



## 策略熵监控

| 统计量 | 健康范围 | 警告信号 | 诊断含义 |
|--------|----------|----------|----------|
| 策略熵均值 | 初始值的 50%~90% | < 初始值的 20% | 熵坍塌，模型丧失探索能力 |
| 策略熵下降速率 | 每 1000 步下降 < 10% | 单步骤降 > 20% | 策略分布急剧收缩 |
| 输出 token 多样性 | 保持较高水平 | 趋近单一 token | 模型退化为确定性输出 |

## 基础设施监控

| 统计量 | 健康范围 | 警告信号 | 诊断含义 |
|--------|----------|----------|----------|
| 显存占用 | < 90% 峰值 | > 95% | OOM 风险 |
| 可训练参数数量 | 与配置一致 | 为 0 或远小于预期 | 参数被静默冻结 |
| 关键层梯度是否全零 | 否 | 连续 3 步全零 | 参数冻结或梯度消失 |
| Checkpoint 文件大小 | 与预期一致 | 为 0 或远小于预期 | 保存过程异常 |

## 综合健康判定规则
  - 绿灯（健康）：所有指标在健康范围内，分位数曲线平稳。
  - 黄灯（警告）：1~2 个指标进入警告区间，持续超过 50 步未恢复。
  - 红灯（危险）：3 个以上指标同时异常，或任一指标出现 NaN/Inf。

️

## 实操建议
- 建立基线：训练前 100~500 步记录各指标分位数，作为"健康基线"。
- 相对判断：后续训练中，若某指标相对基线偏移超过 3~5 倍，触发告警。
- 可视化：在 TensorBoard / WandB 中同时绘制 p5 / p50 / p95 三条曲线，直观观察分布形态变化。
- 自动告警：设置阈值触发告警，避免人工盯盘遗漏。

## 核心原则：
- 监控的目的不是追求某个固定数值，而是尽早发现分布形态的异常变化。三条曲线紧密贴合且平稳 = 健康；三条线发散 = 长尾风险；三条线整体下移趋近零 = 坍缩风险。